# Lecture des six étoiles Druid avec PySpark

Ce notebook lit les six tables **directement dans Druid via JDBC Avatica** et crée un DataFrame Spark par étoile. Les données ne passent pas par Pandas et ne sont pas collectées dans la mémoire Python du pilote.

| Table Druid | DataFrame PySpark |
|---|---|
| `fact_order_item` | `df_fact_order_item` |
| `fact_order` | `df_fact_order` |
| `fact_payment` | `df_fact_payment` |
| `fact_logistics` | `df_fact_logistics` |
| `fact_review` | `df_fact_review` |
| `mart_customer_rfm` | `df_mart_customer_rfm` |

## Avant de commencer

1. Démarrer la stack du dépôt avec le profil `analytics` ou `full`, puis attendre la réussite de `ebrasil-druid-init`.
2. Ouvrir ce notebook dans **un noyau Python neuf** du service `spark-jupyter`.
3. Exécuter les cellules dans l'ordre. Le pilote Avatica **1.28.0**, avec ses dépendances embarquées, est téléchargé depuis Maven au premier démarrage de Spark. Un accès Internet est nécessaire lors de cette résolution; aucun paquet Python supplémentaire n'est installé.

Dans GitHub, le fichier se trouve sous `spark/notebooks/pySpark`. Dans Jupyter, le répertoire racine est déjà `/opt/spark/notebooks` : ouvrir `pySpark/07-Lecture-des-six-etoiles-Druid.ipynb`.

Sur une stack existante, récupérer le fichier avec `git pull origin master`, puis relancer `docker compose run --rm notebooks-init` pour copier les nouveaux notebooks dans le volume Jupyter. Réutiliser les mêmes options `--env-file` que pour le lancement de la stack.


## 1. Paramètres de connexion

Dans le réseau Docker du dépôt, Druid est accessible à **http://druid-router:8888** depuis le pilote et les exécuteurs Spark. Le port **25019** est le port publié sur l'hôte.

Les variables d'environnement facultatives sont `DRUID_URL`, `DRUID_USER`, `DRUID_PASSWORD` et `SPARK_MASTER_URL`. En mode Spark local, sans résolution du nom `druid-router`, l'adresse par défaut est `http://127.0.0.1:25019`. Un cluster distant doit utiliser une adresse Druid accessible depuis **chaque exécuteur**; `localhost` y désignerait l'exécuteur lui-même.


In [1]:
import os
import socket
from urllib.parse import urlsplit

from pyspark import SparkContext, StorageLevel
from pyspark.sql import SparkSession, functions as F

TABLES_DRUID = (
    "fact_order_item",
    "fact_order",
    "fact_payment",
    "fact_logistics",
    "fact_review",
    "mart_customer_rfm",
)
AVATICA_PACKAGE = "org.apache.calcite.avatica:avatica:1.28.0"
AVATICA_DRIVER = "org.apache.calcite.avatica.remote.Driver"

# Paramètres modifiables avant le démarrage de Spark.
SPARK_MASTER_URL = os.environ.get("SPARK_MASTER_URL", "local[2]")
DRUID_URL = os.environ.get("DRUID_URL", "").strip()
if not DRUID_URL:
    try:
        socket.getaddrinfo("druid-router", 8888)
        DRUID_URL = "http://druid-router:8888"
    except socket.gaierror:
        DRUID_URL = "http://127.0.0.1:25019"

parsed_url = urlsplit(DRUID_URL)
if (parsed_url.scheme not in {"http", "https"}
        or not parsed_url.hostname
        or parsed_url.username or parsed_url.password
        or parsed_url.query or parsed_url.fragment
        or parsed_url.path not in {"", "/"}):
    raise ValueError(
        "DRUID_URL doit être une URL de base HTTP(S), par exemple "
        "http://druid-router:8888, sans identifiants ni chemin SQL."
    )
DRUID_URL = DRUID_URL.rstrip("/")
JDBC_URL = (
    "jdbc:avatica:remote:url=" + DRUID_URL
    + "/druid/v2/sql/avatica/;transparent_reconnection=true"
)

FETCH_SIZE = 1000
QUERY_TIMEOUT_SECONDS = 300
CACHE_DATAFRAMES = True


## 2. Démarrer Spark avec le pilote JDBC

La dépendance doit être déclarée **avant le lancement de la JVM Spark**. Si un autre notebook a déjà démarré Spark sans Avatica, redémarrer le noyau et recommencer. Arrêter seulement la session ne suffit pas nécessairement à réinitialiser la JVM.

Le notebook réutilise une session existante si le pilote Avatica y est déjà disponible. La configuration du cluster fournie par Docker, notamment l'adresse du pilote `spark-jupyter`, est conservée.


In [2]:
def verifier_pilote_avatica(session):
    """Vérifie le classpath du pilote; la lecture matérialisée vérifiera les exécuteurs."""
    try:
        loader = session._jvm.java.lang.Thread.currentThread().getContextClassLoader()
        loader.loadClass(AVATICA_DRIVER)
    except Exception as exc:
        raise RuntimeError(
            "Le pilote Avatica n'est pas chargé. Redémarrer le noyau Jupyter "
            "et exécuter ce notebook avant toute autre création de session Spark."
        ) from exc


if SparkContext._active_spark_context is not None:
    spark = SparkSession.builder.getOrCreate()
    verifier_pilote_avatica(spark)
else:
    if SparkContext._gateway is not None:
        raise RuntimeError(
            "La JVM Spark est déjà démarrée. Redémarrer le noyau pour charger "
            "Avatica au démarrage, puis réexécuter les cellules dans l'ordre."
        )
    spark = (
        SparkSession.builder
        .appName("08-Lecture-six-etoiles-Druid")
        .master(SPARK_MASTER_URL)
        .config("spark.jars.packages", AVATICA_PACKAGE)
        .config("spark.sql.session.timeZone", "UTC")
        .getOrCreate()
    )
    verifier_pilote_avatica(spark)

# Les timestamps Druid sont lus avec une convention UTC explicite.
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sparkContext.setLogLevel("WARN")

if (parsed_url.hostname in {"localhost", "127.0.0.1", "::1"}
        and not spark.sparkContext.master.startswith("local")):
    raise ValueError(
        "Un cluster Spark distant ne peut pas lire Druid via localhost. "
        "Définir DRUID_URL avec une adresse accessible depuis ses exécuteurs. "
        "Dans le Docker du dépôt : http://druid-router:8888."
    )

print("Spark :", spark.version)
print("Master :", spark.sparkContext.master)
print("Druid :", DRUID_URL)


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /nonexistent/.ivy2.5.2/cache
The jars for the packages stored in: /nonexistent/.ivy2.5.2/jars
org.apache.calcite.avatica#avatica added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2db5b64c-0a87-475d-b2b8-d7c20936aee7;1.0
	confs: [default]
Exception in thread "main" java.io.FileNotFoundException: /nonexistent/.ivy2.5.2/cache/resolved-org.apache.spark-spark-submit-parent-2db5b64c-0a87-475d-b2b8-d7c20936aee7-1.0.xml (No such file or directory)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:293)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:235)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:184)
	at org.apache.ivy.plugins.parser.xml.XmlModuleDescriptorWriter.write(XmlModuleDescriptorWriter.java:

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## 3. Vérifier la présence des six tables

Les métadonnées sont lues via le même pilote JDBC. Seuls les noms des six tables sont ramenés dans Python. Les identifiants d'authentification facultatifs sont lus depuis l'environnement et ne sont pas affichés.


In [3]:
JDBC_OPTIONS = {
    "url": JDBC_URL,
    "driver": AVATICA_DRIVER,
    "fetchsize": str(FETCH_SIZE),
    "queryTimeout": str(QUERY_TIMEOUT_SECONDS),
    "numPartitions": "1",
    "sqlTimeZone": "UTC",
    "useApproximateCountDistinct": "false",
    # Les exemples suivants réalisent les transformations dans Spark.
    "pushDownPredicate": "false",
    "pushDownAggregate": "false",
    "pushDownLimit": "false",
    "pushDownOffset": "false",
    "pushDownTableSample": "false",
}
for environment_name, jdbc_name in (
    ("DRUID_USER", "user"),
    ("DRUID_PASSWORD", "password"),
):
    if os.environ.get(environment_name):
        JDBC_OPTIONS[jdbc_name] = os.environ[environment_name]

noms_sql = ", ".join("'" + name + "'" for name in TABLES_DRUID)
requete_catalogue = (
    'SELECT TABLE_NAME AS "table_name" FROM INFORMATION_SCHEMA.TABLES '
    "WHERE TABLE_SCHEMA = 'druid' AND TABLE_NAME IN (" + noms_sql + ")"
)

try:
    catalogue_druid = (
        spark.read.format("jdbc")
        .options(**JDBC_OPTIONS)
        .option("query", requete_catalogue)
        .load()
    )
    tables_presentes = {row["table_name"] for row in catalogue_druid.collect()}
except Exception as exc:
    raise RuntimeError(
        "Connexion JDBC à Druid impossible. Vérifier druid-router:8888, "
        "l'accès Avatica, les identifiants éventuels et le classpath Spark."
    ) from exc

tables_manquantes = set(TABLES_DRUID) - tables_presentes
if tables_manquantes:
    raise RuntimeError(
        "Tables Druid absentes ou non visibles : "
        + ", ".join(sorted(tables_manquantes))
        + ". Attendre la réussite de ebrasil-druid-init et vérifier les droits."
    )
print("Les six tables sont visibles dans Druid.")


RuntimeError: Connexion JDBC à Druid impossible. Vérifier druid-router:8888, l'accès Avatica, les identifiants éventuels et le classpath Spark.

## 4. Charger chaque table dans un DataFrame PySpark

La lecture conserve les colonnes et les types exposés par JDBC, y compris sur une table vide. Elle n'ajoute ni limite de lignes ni filtre temporel : le périmètre correspond à toutes les lignes visibles dans Druid au moment des requêtes.

La création d'un DataFrame est paresseuse. La cellule de matérialisation suivante lance réellement les lectures. Une connexion de lecture par table évite de surcharger le Broker dans cette stack de formation. Les transformations Spark peuvent ensuite être repartitionnées si nécessaire.


In [ ]:
def lire_table_druid(nom_table):
    """Lit une étoile complète de la liste autorisée via le connecteur JDBC Spark."""
    if nom_table not in TABLES_DRUID:
        raise ValueError("Table hors du périmètre des six étoiles : " + nom_table)
    try:
        return (
            spark.read.format("jdbc")
            .options(**JDBC_OPTIONS)
            .option("dbtable", '"druid"."' + nom_table + '"')
            .load()
        )
    except Exception as exc:
        raise RuntimeError("Échec de lecture du schéma Druid : " + nom_table) from exc


# Nettoyer uniquement les caches créés par une précédente exécution de cette cellule.
for ancien_dataframe in globals().get("dataframes_druid", {}).values():
    ancien_dataframe.unpersist()

# Une variable distincte par étoile.
df_fact_order_item = lire_table_druid("fact_order_item")
df_fact_order = lire_table_druid("fact_order")
df_fact_payment = lire_table_druid("fact_payment")
df_fact_logistics = lire_table_druid("fact_logistics")
df_fact_review = lire_table_druid("fact_review")
df_mart_customer_rfm = lire_table_druid("mart_customer_rfm")

dataframes_druid = {
    "fact_order_item": df_fact_order_item,
    "fact_order": df_fact_order,
    "fact_payment": df_fact_payment,
    "fact_logistics": df_fact_logistics,
    "fact_review": df_fact_review,
    "mart_customer_rfm": df_mart_customer_rfm,
}


In [ ]:
# Les six count() déclenchent les lectures; aucune ligne métier n'est collectée en Python.
# MEMORY_AND_DISK évite de relire Druid à chaque exemple et autorise un débordement sur disque.
resume_chargement = []
try:
    for nom_table, dataframe in dataframes_druid.items():
        if CACHE_DATAFRAMES:
            dataframe.persist(StorageLevel.MEMORY_AND_DISK)
        nombre_lignes = dataframe.count()
        dataframe.createOrReplaceTempView(nom_table)
        resume_chargement.append((nom_table, nombre_lignes, len(dataframe.columns)))
        print(f"{nom_table} : {nombre_lignes:,} lignes, {len(dataframe.columns)} colonnes")
except Exception as exc:
    for dataframe in dataframes_druid.values():
        dataframe.unpersist()
    raise RuntimeError(
        "Lecture interrompue. Vérifier l'accès Druid depuis les exécuteurs Spark, "
        "le pilote Avatica sur ces exécuteurs et les limites/temps de requête du Broker. "
        "Reprendre à la cellule de chargement des six DataFrames."
    ) from exc

spark.createDataFrame(
    resume_chargement,
    "table_druid string, nombre_lignes long, nombre_colonnes int",
).show(truncate=False)


## 5. Consulter les schémas et un aperçu

Les aperçus sont limités à trois lignes, **pas les DataFrames chargés**. L'affichage vertical convient aux étoiles dénormalisées qui contiennent beaucoup de colonnes. Les lignes d'aperçu n'ont pas d'ordre garanti.


### Ventes — fact_order_item


In [ ]:
df_fact_order_item.printSchema()
df_fact_order_item.show(3, truncate=60, vertical=True)


### Commandes — fact_order


In [ ]:
df_fact_order.printSchema()
df_fact_order.show(3, truncate=60, vertical=True)


### Paiements — fact_payment


In [ ]:
df_fact_payment.printSchema()
df_fact_payment.show(3, truncate=60, vertical=True)


### Logistique — fact_logistics


In [ ]:
df_fact_logistics.printSchema()
df_fact_logistics.show(3, truncate=60, vertical=True)


### Satisfaction — fact_review


In [ ]:
df_fact_review.printSchema()
df_fact_review.show(3, truncate=60, vertical=True)


### CRM / RFM — mart_customer_rfm


In [ ]:
df_mart_customer_rfm.printSchema()
df_mart_customer_rfm.show(3, truncate=60, vertical=True)


## 6. Réutiliser les DataFrames et les vues Spark SQL

Les noms `df_fact_order_item`, `df_fact_order`, `df_fact_payment`, `df_fact_logistics`, `df_fact_review` et `df_mart_customer_rfm` restent disponibles dans le noyau. Le dictionnaire `dataframes_druid` permet aussi d'accéder à une étoile par son nom.

Les vues temporaires ci-dessous sont locales à la session Spark. Elles ne créent ni ne modifient de table dans Druid.


In [ ]:
# Exemple d'API DataFrame sur l'étoile Commandes.
df_fact_order.groupBy("status").count().orderBy(F.desc("count")).show()

# Le même type de lecture via une vue temporaire Spark SQL.
spark.sql("""
    SELECT status, COUNT(*) AS nombre_commandes
    FROM fact_order
    GROUP BY status
    ORDER BY nombre_commandes DESC
""").show()


## Fin de séance et limites de lecture

Pour libérer le cache tout en gardant les variables DataFrame, exécuter :
```python
for dataframe in dataframes_druid.values():
    dataframe.unpersist()
```
À exécuter uniquement en fin de séance : `spark.stop()`. Redémarrer ensuite le noyau avant de recréer une JVM avec d'autres dépendances.

- Les six requêtes ne constituent pas un snapshot transactionnel commun. Éviter une réingestion pendant le chargement; relancer l'ensemble si les tables ont changé.
- Le cache réduit les relectures mais ne garantit pas un snapshot durable en cas de perte d'exécuteur. Pour un snapshot d'analyse reproductible, écrire une copie datée dans le lakehouse.
- Les DataFrames reflètent les types et le grain **réellement ingérés dans Druid**. Les rôles de date dénormalisés peuvent être des chaînes; les convertir explicitement avant des calculs calendaires.
- `fact_payment` provient d'un grain commande × moyen de paiement préparé; `fact_review` porte des comptes agrégés par commande. `fact_logistics` utilise un vendeur représentatif et RFM ne porte que les commandes livrées. Ne pas multiplier les mesures par des jointures entre étoiles sans contrôler les clés.
- Si Druid applique un rollup à l'ingestion, ses lignes ne sont pas nécessairement les événements sources : contrôler la configuration avant d'interpréter un `count()`.
- En cas de `ClassNotFoundException`, redémarrer le noyau et charger Avatica avant Spark. En cas d'échec Maven, vérifier l'accès Internet ou préinstaller le JAR Avatica dans l'image Spark. Un port joignable depuis le navigateur n'établit pas l'accès depuis les exécuteurs.

Références : [JDBC Druid](https://druid.apache.org/docs/latest/querying/sql-jdbc.html/), [JDBC Spark 4.0.4](https://spark.apache.org/docs/4.0.4/sql-data-sources-jdbc.html), [pilote Avatica embarqué](https://calcite.apache.org/avatica/docs/custom_client_artifacts.html), [versions Avatica](https://calcite.apache.org/avatica/downloads/avatica.html).
